# Resave images as tiff (```.msr``` Version) 

## Input

This recipe expects an input folder containing a `raw` folder with ```.msr``` files.

**NOTE:** single channel images are untested at the moment, should work though.

## Output

For each channel of each ```.msr``` image in the input folder, a ```.tif``` file will be created in the output directory. The name of output files will be ```{input file name (without ending)}_ch{channel number}.tif```

## TODO

- support grouped stacks in newer versions of `msr-reader`

In [ ]:
import os
import sys
import json
from glob import glob
import numpy as np
from natsort import natsorted
from pathlib import Path

import tifffile
from msr_reader import OBFFile
from calmutils.imageio import save_tiff_imagej
from nd2_resaving import _as_list


################################################
# reads all msr files and resaves them as tif 
def resave_msr_folder(in_path, out_path=None, file_pattern='[!.]*.msr', exclude_pattern=None, stack_idxs=None):

    files = {f for pat in _as_list(file_pattern) for f in Path(in_path).glob(pat)}
    excluded = {f for pat in _as_list(exclude_pattern) for f in Path(in_path).glob(pat)}
    files = natsorted(files - excluded)

    # by default, save in "tif" subfolder
    if out_path is None:
        out_path = Path(in_path) / 'tif'
    # make out_path a pathlib path in any case
    out_path = Path(out_path)

    if not out_path.exists():
        out_path.mkdir(parents=True)

    for file in files:
        resave_msr_single(file, out_path, stack_idxs=stack_idxs)


def resave_msr_single(in_file, out_path, stack_idxs=None):

    with OBFFile(in_file) as fd:

        # union of all stack idxs and the (optional) subset we want
        idxs_to_load = set(range(fd.num_stacks))
        if stack_idxs is not None:
            idxs_to_load &= set(stack_idxs)

        for idx in idxs_to_load:
            
            # load
            img = fd.read_stack(idx) 
            pixel_size = fd.pixel_size(idx)

            # make 1D "images" (e.g. line profiles) pseudo-2D,
            # otherwise TIFF saving fails
            if img.ndim == 1:
                img = img[np.newaxis]

            # to uint16 (TODO: make settable / optional?)
            img = img.astype(np.uint16)
            
            # save
            stem, _ = os.path.splitext(os.path.basename(in_file))
            out_file = os.path.join(out_path, f"{stem}_ch{idx}.tif")

            save_tiff_imagej(out_file, img, pixel_size=pixel_size, distance_unit='m')



In [13]:
# specify the upper level folder
in_path = '/Users/david/Desktop/scratch_data/msr-test-files'

# subfolder (e.g. "raw") containing nd2 files, leave blank if the files are in in_path
image_subfolder = ''

# folder to save results to
out_subfolder = 'tif'

# glob pattern(s) selecting the raw files to resave (list of patterns allowed)
file_pattern = '[!.]*.msr'

# glob pattern(s) to exclude from file_pattern (e.g. handled in another round)
exclude_pattern = None

# stack idxs to include (default: None -> include all)
stack_idxs = None

In [22]:
resave_msr_folder(
    os.path.join(in_path, image_subfolder),
    file_pattern=file_pattern,
    exclude_pattern=exclude_pattern,
    out_path=os.path.join(in_path, out_subfolder),
    stack_idxs=stack_idxs
)